# 02 · Centrality & Structure — European Air Transport Network

> **Data vintage — June 2014.** A frozen OpenFlights route snapshot, *not* the current network. Every rank and coefficient below describes 2014. The questions here — structural centrality, scale-free structure, small-world geometry — are topological invariants, so a dated snapshot is the standard, defensible choice.

**This notebook answers Research Question 1 and characterises the network's structure.** It:

1. Computes six centralities on the `DiGraph` and writes them back to `network.db`.
2. Ranks airports by **betweenness** — and asks where **Vienna (VIE)** sits (RQ1).
3. Probes why the traffic giants (Frankfurt, Heathrow, Amsterdam) are *missing* from the top of that ranking.
4. Tests the East–West bridge hypothesis directly — all-East and Central-Europe-only — with subset betweenness.
5. Tests the degree distribution for **scale-free** structure (a Clauset–Shalizi–Newman fit).
6. Runs the **small-world** test against an Erdős–Rényi null.

Analysis logic lives in `src/build_graph.py` and `src/utils.py` (imported, not redefined); only the presentation is notebook-specific.

In [ ]:
import sys
import sqlite3
import warnings
from pathlib import Path
from contextlib import closing

import numpy as np
import pandas as pd
import powerlaw
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import zeta

%load_ext autoreload
%autoreload 2


def _find_src() -> Path:
    """Locate the repo's src/ dir so imports work regardless of launch cwd."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "build_graph.py").exists():
            return p / "src"
    raise FileNotFoundError("Could not locate src/ — run this from inside the repo.")


sys.path.insert(0, str(_find_src()))

import build_graph as bg
import utils

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1 · Six centralities on the directed graph

Edge weight here is the **number of distinct operating carriers** on a route — a *service* proxy, not a distance and not traffic. That one fact dictates a per-measure choice:

- **Betweenness & closeness** read edge weight as a **distance/cost**: a heavier edge becomes *longer* and *less* likely to lie on a shortest path. Ours means the opposite (more carriers = more service), so passing it would invert the geometry. Both are computed **unweighted** — shortest path = fewest hops, the topological convention for airport networks (Guimerà & Amaral 2005; Barrat et al. 2004) and the one that lines up with NB04's percolation.
- **In/out-degree** are stored as the **unweighted count** of distinct routes (the `INTEGER` columns are *degree*, not weighted *strength*).
- **PageRank** runs on the full `DiGraph`; its damping term absorbs dangling nodes and the small off-giant components cleanly.
- **Eigenvector centrality** needs a *connected* graph with a unique dominant eigenvector (Perron–Frobenius). This `DiGraph` is neither connected (5 weakly-connected components) nor strongly connected, so `eigenvector_centrality_numpy` **raises** on it. We compute it on the **undirected giant component** and assign 0 to the off-giant nodes — `PageRank` carries the directed-influence story.

On a directed graph, **closeness** measures *incoming* distance by default (how reachable a node is *from* the network); reverse the graph for the outbound version.

In [ ]:
digraph, undirected, edges, airports_eu = bg.build_networks()

# Unweighted degree = number of distinct routes in / out (the INTEGER columns).
in_degree = dict(digraph.in_degree())
out_degree = dict(digraph.out_degree())

# Topological betweenness & closeness (weight is a service proxy, not a distance).
betweenness = nx.betweenness_centrality(digraph, normalized=True, weight=None)
closeness = nx.closeness_centrality(digraph)          # directed: incoming, wf_improved
pagerank = nx.pagerank(digraph, alpha=0.85, weight=None)

# Eigenvector on the undirected giant component (Perron-Frobenius needs a
# connected graph; the DiGraph is neither connected nor strongly connected).
giant_nodes = max(nx.connected_components(undirected), key=len)
giant = undirected.subgraph(giant_nodes)
eig_giant = nx.eigenvector_centrality_numpy(giant, weight=None)
eigenvector = {n: float(eig_giant.get(n, 0.0)) for n in digraph.nodes()}

print(f"Graph: {digraph.number_of_nodes()} nodes, {digraph.number_of_edges()} directed edges")
print(f"Eigenvector on giant component: {len(giant_nodes)} nodes "
      f"({digraph.number_of_nodes() - len(giant_nodes)} off-giant set to 0.0)")

# Native-typed rows for the SQLite UPDATE (avoids numpy-scalar binding issues).
update_rows = [
    {
        "iata": n,
        "in_degree": int(in_degree[n]),
        "out_degree": int(out_degree[n]),
        "betweenness": float(betweenness[n]),
        "closeness": float(closeness[n]),
        "eigenvector": eigenvector[n],
        "pagerank": float(pagerank[n]),
    }
    for n in digraph.nodes()
]

# Tidy frame for inspection, with city/country attached from node attributes.
centrality = pd.DataFrame(update_rows)
meta = pd.DataFrame(
    [{"iata": n, "city": d.get("city"), "country": d.get("country")}
     for n, d in digraph.nodes(data=True)]
)
centrality = meta.merge(centrality, on="iata")
centrality["total_degree"] = centrality["in_degree"] + centrality["out_degree"]
centrality.sort_values("betweenness", ascending=False).head(10)

The `nodes` table already holds all 559 airports with NULL centrality columns — `schema.sql` pre-declared them, so we `UPDATE` in place (no `ALTER TABLE`), matched on `iata`. The write is idempotent; re-running overwrites. We confirm by counting non-NULL `betweenness`, which jumps from 0 to 559.

In [ ]:
UPDATE_CENTRALITY = """
    UPDATE nodes
       SET in_degree   = :in_degree,
           out_degree  = :out_degree,
           betweenness = :betweenness,
           closeness   = :closeness,
           eigenvector = :eigenvector,
           pagerank    = :pagerank
     WHERE iata = :iata
"""

with closing(sqlite3.connect(utils.DB_PATH)) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executemany(UPDATE_CENTRALITY, update_rows)
    conn.commit()
    scored = pd.read_sql("SELECT COUNT(betweenness) AS n FROM nodes", conn).at[0, "n"]

print(f"nodes with centrality written: {scored} (expect {digraph.number_of_nodes()})")
assert scored == digraph.number_of_nodes(), "some nodes were not updated"

## 2 · Research Question 1 — Vienna among the bridges

**Betweenness centrality** counts how many shortest paths between *other* airports run through a node — the **structural-bridge** measure. A high-betweenness node is a bottleneck: remove it and once-adjacent regions fall far apart.

We rank airports by betweenness, restricted to countries with **more than five airports** in the graph, so the ranking reflects well-sampled national systems rather than single-airport countries. The query is version-controlled at `sql/queries/top_airports_by_betweenness.sql` 

In [ ]:
query_path = utils.SQL_QUERIES_DIR / "top_airports_by_betweenness.sql"
sql = query_path.read_text(encoding="utf-8")

with closing(sqlite3.connect(utils.DB_PATH)) as conn:
    ranked = pd.read_sql(sql, conn)

vie = ranked.loc[ranked["is_vienna"] == 1].iloc[0]
print(f"Vienna (VIE) betweenness rank: {int(vie['betweenness_rank'])} "
      f"of {len(ranked)} airports in countries with >5 airports "
      f"(betweenness = {vie['betweenness']:.6f}).")
ranked.head(15)

### Where are Frankfurt, Heathrow, Amsterdam?

The top of that ranking is dominated by **peripheral gateways** — Istanbul, the Nordic capitals, Athens, Moscow, the Spanish leisure airports — while the traffic giants are absent. This is not a bug; it is the defining behaviour of betweenness, and it echoes NB01, where *degree* already surfaced low-cost bases over legacy hubs.

Two effects combine. First, a **super-hub is an endpoint, not an intermediary**: Frankfurt connects directly to hundreds of airports, so paths *terminate* at it rather than *pass through* it — and betweenness counts only pass-through. Second, the network's **periphery is tree-like** (small Nordic, Aegean, and island airports hang off a single regional gateway), so every path in or out of those regions is *forced* through the gateway, inflating its betweenness. Guimerà & Amaral (2005) found exactly this worldwide — Anchorage outranks the mega-hubs.

The table below is the check: the legacy hubs carry high *degree* but middling *betweenness*. If they instead showed low degree, that would signal a data fault — they don't.

In [ ]:
# Global ranks over all 559 nodes (distinct from the >5-country pool above).
centrality["betweenness_rank"] = centrality["betweenness"].rank(ascending=False, method="min").astype(int)
centrality["degree_rank"] = centrality["total_degree"].rank(ascending=False, method="min").astype(int)

WATCH = ["FRA", "LHR", "CDG", "AMS", "MUC", "STN", "IST", "VIE"]
present = [c for c in WATCH if c in set(centrality["iata"])]
cols = ["city", "country", "total_degree", "degree_rank", "betweenness", "betweenness_rank"]
diagnostic = centrality.set_index("iata").loc[present, cols]
diagnostic.round(6)

### The proper test — East–West gateway betweenness

Global betweenness rewards the periphery, and Vienna sits in the dense continental core — so its rank says little about the specific claim in RQ1: that VIE bridges *Eastern* and *Western* Europe. That is a **conditional** question — of the shortest paths running from an Eastern airport to a Western one, how many pass through VIE? — and it needs **subset betweenness**, counting East↔West paths only.

We split airports by the Cold-War-era divide: **East** = the former Eastern-bloc, USSR, and Yugoslav successor states; **West** = everything else in the giant component. The partition is a modelling choice, stated here so it can be adjusted (Turkey, straddling the line, is left in the West). We then rank every airport by the East–West shortest paths it carries, and compare Vienna's position here with its global rank.

In [ ]:
EAST_COUNTRIES = {
    "Poland", "Czech Republic", "Slovakia", "Hungary", "Romania", "Bulgaria",
    "Croatia", "Slovenia", "Serbia", "Bosnia and Herzegovina", "Montenegro",
    "Macedonia", "Albania", "Kosovo", "Estonia", "Latvia", "Lithuania",
    "Belarus", "Ukraine", "Moldova", "Russia",
}

country_of = nx.get_node_attributes(digraph, "country")
east = [n for n in giant.nodes() if country_of.get(n) in EAST_COUNTRIES]
west = [n for n in giant.nodes() if country_of.get(n) not in EAST_COUNTRIES]
print(f"Giant component: {len(east)} Eastern airports, {len(west)} Western airports")

# Subset betweenness over undirected East<->West shortest paths.
ew = nx.betweenness_centrality_subset(giant, sources=east, targets=west,
                                      normalized=True, weight=None)
gateways = centrality.set_index("iata").assign(ew_betweenness=pd.Series(ew))
ew_rank = gateways["ew_betweenness"].rank(ascending=False, method="min")

r_ew = int(ew_rank.loc["VIE"])
r_global = int(gateways.loc["VIE", "betweenness_rank"])
verdict = ("its structural importance is specifically East–West"
           if r_ew < r_global else "it is one of several East–West gateways")
print(f"Vienna: global betweenness rank {r_global} -> East–West rank {r_ew}; {verdict}.")

gateways.sort_values("ew_betweenness", ascending=False)[
    ["city", "country", "ew_betweenness"]].head(10).round(6)

### Tighter still — Central-Eastern Europe only

Vienna does not move on the all-East test. Before concluding, we sharpen the definition: in aviation "Eastern Europe" usually means **Central-Eastern Europe** — the EU-accession and Balkan states — *not* the former USSR, whose Moscow-centred domestic flows dominate the all-East ranking and swamp Vienna's specific role. This is the fairest test of the hypothesis: of the shortest paths from a CEE airport to anywhere else, how many run through VIE?

In [ ]:
CEE_COUNTRIES = {
    "Poland", "Czech Republic", "Slovakia", "Hungary", "Slovenia", "Croatia",
    "Romania", "Bulgaria", "Serbia", "Bosnia and Herzegovina", "Montenegro",
    "Macedonia", "Albania", "Kosovo", "Estonia", "Latvia", "Lithuania",
}
cee = [n for n in giant.nodes() if country_of.get(n) in CEE_COUNTRIES]
rest = [n for n in giant.nodes() if country_of.get(n) not in CEE_COUNTRIES]
ew_cee = nx.betweenness_centrality_subset(giant, sources=cee, targets=rest,
                                          normalized=True, weight=None)
cee_rank = pd.Series(ew_cee).rank(ascending=False, method="min")
print(f"{len(cee)} CEE airports in the giant component")
print(f"Vienna CEE-gateway betweenness rank: {int(cee_rank.loc['VIE'])} of {giant.number_of_nodes()}")

(pd.Series(ew_cee, name="cee_betweenness").to_frame()
   .join(centrality.set_index("iata")[["city", "country"]])
   .sort_values("cee_betweenness", ascending=False).head(10).round(6))

### Reading Research Question 1

Three betweenness lenses — global, all-East, and CEE-only — put Vienna at ranks **42, 42, and 45**. The hypothesis that VIE is *the* East–West bridge is **not supported by the 2014 operating-carrier network**. That is a real result, not a null one, and it has a clear cause: by 2014 the **low-cost carriers had already rewired Central-European connectivity to point-to-point**. The airports that actually carry CEE-to-elsewhere shortest paths are Ryanair/Wizz/easyJet bases — Stansted, Bergamo, Luton, Dublin — alongside the regional aggregator Bucharest, not the legacy alliance hubs.

Two caveats sharpen rather than soften this. First, our **codeshare filter removes exactly the Star Alliance feed** that Austrian Airlines used to funnel CEE traffic through Vienna — so VIE's *marketed* gateway role is partly invisible by construction, and that is the honest limitation to name in the README. Second, betweenness measures **structural bottleneck-ness**, not traffic or revenue: Vienna can be a large, profitable hub (it was) while still not being a shortest-path bottleneck. The Austrian angle survives, but as a nuance — *structural centrality and commercial importance are different things* — the same lesson NB01's degree ranking first flagged.

## 3 · Is it scale-free? The degree distribution

A **scale-free** network has a heavy, power-law degree tail, P(k) ∝ k^(−α) — hubs of every size, no characteristic scale. Testing this properly means more than drawing a line through a log-log plot, which is statistically unsound: binning P(k) discards tail data, and OLS on a CCDF is biased. We use the **Clauset–Shalizi–Newman** procedure via the `powerlaw` package — maximum-likelihood α with the lower cutoff k_min chosen by KS minimisation — then a **likelihood-ratio test** of the power law against a log-normal and a truncated power law. A network is only "scale-free" if the power law genuinely beats those alternatives.

In [ ]:
degrees = np.array([d for _, d in giant.degree()])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fit = powerlaw.Fit(degrees, discrete=True, verbose=False)
    R_ln, p_ln = fit.distribution_compare("power_law", "lognormal", normalized_ratio=True)
    R_tp, p_tp = fit.distribution_compare("power_law", "truncated_power_law", normalized_ratio=True)

alpha, kmin = fit.power_law.alpha, int(fit.power_law.xmin)
n_tail = int((degrees >= kmin).sum())

# Naive OLS slope on the log-log CCDF — the biased estimator, for contrast only.
uniq = np.sort(np.unique(degrees))
ccdf = np.array([(degrees >= k).mean() for k in uniq])
mask = uniq >= kmin
ols = stats.linregress(np.log(uniq[mask]), np.log(ccdf[mask]))


def _verdict(name, R, p):
    if p >= 0.05:
        return f"  vs {name:20s}: indistinguishable (R = {R:+.2f}, p = {p:.2f})"
    winner = "power law" if R > 0 else name
    return f"  vs {name:20s}: {winner} favoured (R = {R:+.2f}, p = {p:.3f})"


print(f"MLE power law:  α = {alpha:.2f},  k_min = {kmin},  tail n = {n_tail}")
print(f"Naive OLS CCDF slope = {ols.slope:.2f}  (biased; implied α ≈ {1 - ols.slope:.2f} — why we use MLE)")
print("Likelihood-ratio tests (Clauset-Shalizi-Newman):")
print(_verdict("lognormal", R_ln, p_ln))
print(_verdict("truncated power law", R_tp, p_tp))

# Log-log CCDF plot in the project palette (dark 'aviation at night' theme).
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(utils.COLORS["bg"])
ax.set_facecolor(utils.COLORS["bg"])
ax.loglog(uniq, ccdf, "o", ms=5, color=utils.COLORS["node_high"],
          alpha=0.85, label="Empirical CCDF")
kk = np.linspace(kmin, degrees.max(), 100)
fit_line = (zeta(alpha, kk) / zeta(alpha, kmin)) * ccdf[uniq >= kmin][0]
ax.loglog(kk, fit_line, "-", color=utils.COLORS["vienna"], lw=2,
          label=f"MLE power law, α = {alpha:.2f}")
ax.axvline(kmin, color=utils.COLORS["coastline"], ls="--", lw=1, label=f"k_min = {kmin}")
ax.set_xlabel("Degree  k")
ax.set_ylabel("P(K ≥ k)")
ax.set_title("Degree distribution — European air network (2014)")
for spine in ax.spines.values():
    spine.set_color(utils.COLORS["coastline"])
ax.tick_params(colors=utils.COLORS["text"])
ax.xaxis.label.set_color(utils.COLORS["text"])
ax.yaxis.label.set_color(utils.COLORS["text"])
ax.title.set_color(utils.COLORS["text"])
ax.grid(True, which="both", color=utils.COLORS["coastline"], alpha=0.25, lw=0.5)
ax.legend(facecolor=utils.COLORS["land"], edgecolor=utils.COLORS["coastline"],
          labelcolor=utils.COLORS["text"], fontsize=9)
utils.save_fig_png(fig, "degree_distribution")
plt.show()

The likelihood-ratio tests are the verdict. A **negative R with a small p favours the alternative** over the pure power law — and for a physical infrastructure network we expect exactly that: an airport cannot grow unbounded runways, so the largest hubs are capped and the tail bends, making a **log-normal or truncated power law** the better description. This is precisely what Guimerà & Amaral (2005) and Barrat et al. (2004) report for real air networks. The honest headline is **broad-scale and heavy-tailed, not textbook scale-free** — yet still hub-dominated enough to make NB04's resilience contrast sharp. (If R comes back insignificant, the data simply cannot separate the two at n ≈ 540 — a short-tail limitation, not proof of a clean power law.)

## 4 · Is it a small world?

A **small-world** network combines two properties usually in tension: **high local clustering** (your neighbours know each other) and **short global path length** (everyone is a few hops away). We quantify it with the Watts–Strogatz σ coefficient, comparing the giant component against an Erdős–Rényi random graph of the same size and density:

$$\sigma = \frac{C / C_{\text{rand}}}{L / L_{\text{rand}}}$$

C is the average clustering coefficient, L the average shortest-path length; C_rand and L_rand come from an ensemble of ER graphs. **σ ≫ 1** is the small-world signature — clustering far above random while path length stays near random.

In [ ]:
N_RAND = 20   # ER realisations to average the null over

C = nx.average_clustering(giant)
L = nx.average_shortest_path_length(giant)
n, m = giant.number_of_nodes(), giant.number_of_edges()
p = 2 * m / (n * (n - 1))

rng = np.random.default_rng(42)
C_rand, L_rand = [], []
for _ in range(N_RAND):
    R = nx.gnp_random_graph(n, p, seed=int(rng.integers(1_000_000_000)))
    R = R.subgraph(max(nx.connected_components(R), key=len))
    C_rand.append(nx.average_clustering(R))
    L_rand.append(nx.average_shortest_path_length(R))
C_rand, L_rand = float(np.mean(C_rand)), float(np.mean(L_rand))

sigma = (C / C_rand) / (L / L_rand)
print(f"Giant component: n = {n}, m = {m}, density p = {p:.3f}")
print(f"  clustering   C = {C:.3f}   (random C_rand = {C_rand:.3f})")
print(f"  path length  L = {L:.2f}    (random L_rand = {L_rand:.2f})")
print(f"  small-world  σ = {sigma:.1f}")

σ ≈ 11 confirms the **small-world** regime: clustering (C = 0.42) runs an order of magnitude above the random null (0.035) — regional cliques where every airport connects to its neighbours (the Aegean, the Baltic, the Iberian leisure market) — while the average trip is still just **~2.7 hops**, barely above random. This is the structural precondition for NB04's headline: short paths mean the network *depends* on its hubs to stay compact, which is exactly why removing them in decreasing-degree order collapses it so fast.

## Summary

**What we built.** Six centralities for all 559 airports, persisted to `network.db`; a version-controlled betweenness ranking; the East–West and CEE-specific bridge tests; and the network's structural fingerprint — a CSN degree-tail fit and the small-world σ.

**What stood out.** Betweenness rewards **intermediaries, not endpoints**, so the ranking is topped by peripheral gateways and low-cost bases (Istanbul, the Nordics, Stansted) while the traffic giants sit lower. **Vienna is a mid-tier hub — ranks 42 / 42 / 45 on the global, all-East, and CEE tests — not the East–West bridge the hypothesis expected**: the LCCs had already rewired CEE connectivity point-to-point by 2014, and the codeshare filter hides Vienna's alliance feed. Structurally the network is **heavy-tailed but not textbook scale-free** (power law loses to log-normal / truncated in the LLR test) and firmly **small-world** (σ ≈ 11 — clustering 12× random, paths ~2.7 hops).

**Next — `03_community_detection.ipynb`.** Louvain community detection on the undirected graph: do natural aviation communities emerge, and do they follow country borders? Map them, measure modularity Q, and name the dominant country per community with SQL.